[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nahuelcostac/generative-networks/blob/main/pretraining/contrastive_learning.ipynb)

This notebook is part of the [Generative Networks course](https://www.nahuelcosta.com/teaching/generative-networks).

# Contrastive Learning with SimCLR

We construct two augmented views of every CIFAR-10 image, train an encoder and projection head with the symmetric NT-Xent objective, discard the projection head, and evaluate the frozen backbone.

The model in this notebook is deliberately small. It demonstrates a pretraining objective and representation-transfer workflow; it is **not** a foundation model. Foundation models additionally require broad data, much larger capacity, and extensive compute.


The projection head is not a cosmetic detail. SimCLR applies its contrastive loss to projected vectors `z`, while downstream tasks use the backbone representation `h`. The augmentations define which changes the representation is encouraged to ignore.


## 0. Dependencies


In [ ]:
from importlib import util
import subprocess
import sys

REQUIRED = {"torch": "torch", "torchvision": "torchvision", "matplotlib": "matplotlib", "numpy": "numpy"}
missing = [package for module, package in REQUIRED.items() if util.find_spec(module) is None]
if missing:
    print("Installing missing course dependencies:", ", ".join(missing))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
else:
    print("Course dependencies are already installed.")


## 1. Imports and Reproducibility


In [ ]:
import copy
import math
import os
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, Dataset, Subset, TensorDataset
from torchvision import datasets, transforms
from torchvision.utils import make_grid

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
fast_dev_run = os.environ.get("FAST_DEV_RUN", "0") == "1"
print(f"Using device: {device}; FAST_DEV_RUN={fast_dev_run}")


## 2. Configuration


In [ ]:
project_root = Path.cwd().parent if Path.cwd().name == "pretraining" else Path.cwd()
output_dir = project_root / "pretraining" / "outputs" / "contrastive_learning"
output_dir.mkdir(parents=True, exist_ok=True)
checkpoint_path = output_dir / "simclr_encoder.pt"
batch_size = 64 if fast_dev_run else 256
epochs = 1 if fast_dev_run else 25
max_steps = 2 if fast_dev_run else None
temperature = 0.2
learning_rate = 1e-3
print("Artifacts:", output_dir)


## 3. CIFAR-10 and Two Correlated Views


In [ ]:
CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD = (0.2470, 0.2435, 0.2616)
data_root = project_root / "data"

evaluation_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
])
raw_train = datasets.CIFAR10(data_root, train=True, download=True)
probe_source = datasets.CIFAR10(data_root, train=True, transform=evaluation_transform)
test_source = datasets.CIFAR10(data_root, train=False, download=True, transform=evaluation_transform)

generator = torch.Generator().manual_seed(SEED)
permutation = torch.randperm(len(raw_train), generator=generator).tolist()
pretrain_count = 256 if fast_dev_run else 20_000
probe_count = 256 if fast_dev_run else 10_000
test_count = 256 if fast_dev_run else len(test_source)
pretrain_indices = permutation[:pretrain_count]
probe_indices = permutation[:probe_count]

probe_train = Subset(probe_source, probe_indices)
probe_test = Subset(test_source, range(test_count))
loader_kwargs = {"num_workers": 0 if fast_dev_run else 2, "pin_memory": device.type == "cuda"}
probe_train_loader = DataLoader(probe_train, batch_size=256, shuffle=False, **loader_kwargs)
probe_test_loader = DataLoader(probe_test, batch_size=256, shuffle=False, **loader_kwargs)

print(f"Unlabeled pretraining images: {pretrain_count:,}")
print(f"Labeled probe images: {probe_count:,}; test images: {test_count:,}")


In [ ]:
contrastive_transform = transforms.Compose([
    transforms.RandomResizedCrop(32, scale=(0.2, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomApply([transforms.ColorJitter(0.4, 0.4, 0.4, 0.1)], p=0.8),
    transforms.RandomGrayscale(p=0.2),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
])

class TwoViews(Dataset):
    def __init__(self, source, indices):
        self.source, self.indices = source, indices
    def __len__(self):
        return len(self.indices)
    def __getitem__(self, index):
        image, _ = self.source[self.indices[index]]
        return contrastive_transform(image), contrastive_transform(image)

pretrain_loader = DataLoader(
    TwoViews(raw_train, pretrain_indices), batch_size=batch_size, shuffle=True,
    drop_last=True, generator=torch.Generator().manual_seed(SEED), **loader_kwargs,
)
view_a, view_b = next(iter(pretrain_loader))
shown = torch.cat([view_a[:8], view_b[:8]])
shown = shown * torch.tensor(CIFAR_STD).view(1, 3, 1, 1) + torch.tensor(CIFAR_MEAN).view(1, 3, 1, 1)
plt.figure(figsize=(12, 3)); plt.imshow(make_grid(shown.clamp(0, 1), nrow=8).permute(1, 2, 0)); plt.axis("off")
plt.title("Two independently augmented rows from the same eight images")


## 4. Encoder and Projection Head


In [ ]:
class ConvEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Conv2d(3, 64, 3, padding=1, bias=False), nn.BatchNorm2d(64), nn.ReLU(),
            nn.Conv2d(64, 128, 3, stride=2, padding=1, bias=False), nn.BatchNorm2d(128), nn.ReLU(),
            nn.Conv2d(128, 256, 3, stride=2, padding=1, bias=False), nn.BatchNorm2d(256), nn.ReLU(),
            nn.Conv2d(256, 256, 3, stride=2, padding=1, bias=False), nn.BatchNorm2d(256), nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),
        )
    def forward(self, images):
        return self.network(images).flatten(1)


In [ ]:
class SimCLR(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = ConvEncoder()
        self.projector = nn.Sequential(nn.Linear(256, 256), nn.ReLU(), nn.Linear(256, 128))
    def forward(self, images):
        h = self.encoder(images)
        return h, F.normalize(self.projector(h), dim=1)

model = SimCLR().to(device)
print(f"Backbone parameters: {sum(p.numel() for p in model.encoder.parameters()) / 1e6:.2f}M")
print(f"Projection-head parameters: {sum(p.numel() for p in model.projector.parameters()):,}")


## 5. Symmetric NT-Xent

For a batch of `B` images, concatenating both views produces `2B` embeddings. Entry `i` is paired with `i+B` (modulo `2B`); the diagonal is excluded so an embedding cannot select itself.


In [ ]:
def nt_xent(z_a, z_b, tau=temperature):
    batch = z_a.size(0)
    z = torch.cat([z_a, z_b], dim=0)
    logits = z @ z.T / tau
    logits.fill_diagonal_(-torch.inf)
    targets = (torch.arange(2 * batch, device=z.device) + batch) % (2 * batch)
    assert torch.equal(targets[targets], torch.arange(2 * batch, device=z.device))
    return F.cross_entropy(logits, targets), logits, targets

_, z_a = model(view_a[:8].to(device))
_, z_b = model(view_b[:8].to(device))
loss, logits, targets = nt_xent(z_a, z_b)
loss.backward()
assert math.isfinite(loss.item()) and model.projector[-1].weight.grad is not None
model.zero_grad(set_to_none=True)
print("Initial loss:", loss.item(), "positive indices:", targets.tolist())


## 6. Pretrain with Contrastive Pairs


In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=1e-4)
history, step = [], 0
for epoch in range(epochs):
    model.train(); running = []
    for view_a, view_b in pretrain_loader:
        view_a, view_b = view_a.to(device), view_b.to(device)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=device.type == "cuda"):
            _, z_a = model(view_a); _, z_b = model(view_b)
            loss, _, _ = nt_xent(z_a, z_b)
        optimizer.zero_grad(set_to_none=True); loss.backward(); optimizer.step()
        running.append(loss.item()); step += 1
        if max_steps is not None and step >= max_steps: break
    history.append(float(np.mean(running)))
    print(f"Epoch {epoch + 1:02d}/{epochs:02d} | NT-Xent={history[-1]:.4f}")
    if max_steps is not None and step >= max_steps: break

torch.save({"model": model.state_dict(), "history": history, "temperature": temperature}, checkpoint_path)
assert np.isfinite(history).all()
plt.plot(range(1, len(history) + 1), history, marker="o"); plt.xlabel("Epoch"); plt.ylabel("NT-Xent")
plt.title("Contrastive pretraining"); plt.grid(alpha=0.3)


## 7. Inspect the Similarity Matrix


In [ ]:
model.eval()
view_a, view_b = next(iter(pretrain_loader))
with torch.no_grad():
    _, z_a = model(view_a[:16].to(device)); _, z_b = model(view_b[:16].to(device))
    similarities = (z_a @ z_b.T).cpu()
positive_similarity = similarities.diag().mean().item()
negative_similarity = similarities[~torch.eye(16, dtype=torch.bool)].mean().item()
print(f"Mean positive cosine similarity: {positive_similarity:.3f}")
print(f"Mean cross-image similarity:     {negative_similarity:.3f}")
plt.figure(figsize=(5, 4)); plt.imshow(similarities, vmin=-1, vmax=1, cmap="coolwarm"); plt.colorbar(label="cosine similarity")
plt.xlabel("second-view index"); plt.ylabel("first-view index"); plt.title("Paired examples lie on the diagonal")


## 8. Discard the Projector and Probe the Backbone


In [ ]:
@torch.no_grad()
def extract_features(encoder, loader):
    encoder.eval()
    features, labels = [], []
    for images, targets in loader:
        features.append(encoder(images.to(device)).float().cpu())
        labels.append(targets)
    return torch.cat(features), torch.cat(labels)

def fit_linear_probe(encoder, name):
    for parameter in encoder.parameters():
        parameter.requires_grad_(False)
    train_x, train_y = extract_features(encoder, probe_train_loader)
    test_x, test_y = extract_features(encoder, probe_test_loader)
    head = nn.Linear(train_x.size(1), 10).to(device)
    optimizer = torch.optim.AdamW(head.parameters(), lr=3e-3, weight_decay=1e-4)
    train_loader = DataLoader(
        TensorDataset(train_x, train_y), batch_size=256, shuffle=True,
        generator=torch.Generator().manual_seed(SEED),
    )
    epochs = 1 if fast_dev_run else 15
    history = []
    for epoch in range(epochs):
        head.train()
        for features, targets in train_loader:
            logits = head(features.to(device))
            loss = F.cross_entropy(logits, targets.to(device))
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()
        head.eval()
        with torch.no_grad():
            predictions = head(test_x.to(device)).argmax(1).cpu()
        accuracy = (predictions == test_y).float().mean().item()
        history.append(accuracy)
    print(f"{name}: final frozen-probe accuracy={accuracy:.3f}")
    return {"name": name, "accuracy": accuracy, "history": history}


In [ ]:
random_encoder = ConvEncoder()
pretrained_encoder = model.encoder


In [ ]:
probe_results = [
    fit_linear_probe(random_encoder.to(device), "Random frozen encoder"),
    fit_linear_probe(pretrained_encoder.to(device), "Pretrained frozen encoder"),
]
plt.figure(figsize=(6, 4))
plt.bar([row["name"] for row in probe_results], [row["accuracy"] for row in probe_results])
plt.ylabel("CIFAR-10 test accuracy")
plt.xticks(rotation=15, ha="right")
plt.ylim(0, 1)
plt.title("Representation quality through a linear probe")
plt.tight_layout()


## 9. Reload the Checkpoint


In [ ]:
reloaded = SimCLR().to(device)
reloaded.load_state_dict(torch.load(checkpoint_path, map_location=device, weights_only=True)["model"])
reloaded.eval(); model.eval()
check = view_a[:4].to(device)
with torch.no_grad():
    assert torch.equal(model.encoder(check), reloaded.encoder(check))
print("Reloaded backbone features match exactly.")


## 10. Things to Try

- Remove color jitter or cropping and inspect what invariance disappears.
- Sweep temperature and batch size together.
- Compare the backbone representation with the projected representation in the linear probe.
- Replace in-batch negatives with a memory queue.

## Troubleshooting

- Contrastive learning benefits from large batches; lowering the batch size changes the number of negatives.
- A two-step smoke run validates indexing but cannot learn useful invariances.
- Do not treat a low contrastive loss alone as proof of downstream quality; use the frozen probe.

**Reference:** [A Simple Framework for Contrastive Learning of Visual Representations](https://arxiv.org/abs/2002.05709).
